# 🚀 Two-Stage Hierarchical SupCon + Linear Probe (NO CRF) trên Vast.ai RTX 3090

**✅ NOTEBOOK ĐỘC LẬP HOÀN TOÀN (SELF-CONTAINED):**
Chỉ cần upload **duy nhất file notebook này** lên Vast.ai và chọn **Run All**!
- Không cần upload thư mục code, không clone Git.
- Toàn bộ logic mô hình H-SupCon, Linear Probe (không CRF), tiền xử lý và đánh giá CoNLL đều nằm trực tiếp trong các cell bên dưới.
- Tự động tải PAP_NER (5.162 câu test dedup) từ Hugging Face Hub.
- Tự động chạy qua 3 Random Seeds `[42, 123, 2024]` và xuất bảng $\mu \pm \sigma$ cho bài báo khoa học.

## Bước 1: Kiểm tra GPU RTX 3090 trên Vast.ai

In [ ]:
!nvidia-smi

## Bước 2: Cài đặt dependencies tự động trên Vast.ai

In [ ]:
import os
import re
import subprocess
import sys
import time

def run_pip(label, requirements, extra_args=()):
    command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
               "--progress-bar", "raw", *extra_args, *requirements]
    print(f"\n=== {label} ===", flush=True)
    print("Lệnh:", " ".join(command), flush=True)
    started = time.monotonic()
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", bufsize=1, env=env)
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    elapsed = time.monotonic() - started
    print(f"=== {label}: {'XONG' if return_code == 0 else 'LỖI'} sau {elapsed:.1f} giây ===", flush=True)
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

print("Kiểm tra PyTorch CUDA...", flush=True)
torch_probe = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.is_available())"],
                             capture_output=True, text=True)
if torch_probe.returncode != 0 or torch_probe.stdout.strip() != "True":
    print("Cài đặt PyTorch CUDA 12.1...", flush=True)
    run_pip("PyTorch CUDA 12.1", ["torch==2.5.1"],
            ["--force-reinstall", "--index-url", "https://download.pytorch.org/whl/cu121"])

run_pip("Thư viện cần thiết", [
    "transformers>=4.28.0", "scikit-learn", "pandas", "numpy",
    "matplotlib", "tqdm", "huggingface_hub>=0.24,<1.0", "prettytable", "py_vncorenlp"
])

import torch
assert torch.cuda.is_available(), "PyTorch không thấy GPU CUDA!"
print(f"✅ CUDA sẵn sàng: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")


## Bước 3: Tải dữ liệu PAP_NER chuẩn từ Hugging Face Hub (Revision cố định)

In [ ]:
from pathlib import Path
import shutil
from huggingface_hub import hf_hub_download

DATA_DIR = Path("./Dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Tải dữ liệu train/dev/test PAP_NER...")
HF_DATA_REPO = "levandai1704/PAP_NER_dedup_70_15_15"
HF_REVISION = "e4af303d22f878e5827c94861f50f747a6b3ed45"

for split, fn in [('train', 'train_data.txt'), ('dev', 'dev_data.txt'), ('test', 'test_data_dedup_70_15_15.txt')]:
    target_file = DATA_DIR / f"{split}.txt"
    if not target_file.exists():
        downloaded = hf_hub_download(repo_id=HF_DATA_REPO, filename=fn, repo_type="dataset", revision=HF_REVISION)
        shutil.copyfile(downloaded, target_file)
        print(f"  ✅ Đã tải {split}.txt ({target_file.stat().st_size:,} bytes)")
    else:
        print(f"  ✅ Đã có sẵn: {target_file}")


## Bước 4: Mã nguồn toàn bộ Mô hình, H-SupCon Loss, Linear Probe (NO CRF) và CoNLL Eval

Toàn bộ code được đặt trực tiếp trong cell này. Không cần import file bên ngoài!

In [ ]:
import os
import sys
import time
import math
import random
import logging
import datetime
from typing import List, Tuple, Union
from collections import defaultdict, namedtuple
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ---------------- 1. CONSTANTS & LABEL MAPPINGS ----------------
LABEL2ID_PAP_NER = {
    'O': 0,
    'B-CQ': 1, 'I-CQ': 2,
    'B-ĐT': 3, 'I-ĐT': 4,
    'B-VBPL': 5, 'I-VBPL': 6,
    'B-NG': 7, 'I-NG': 8,
    'B-SL': 9, 'I-SL': 10
}
ID2LABEL_PAP_NER = {v: k for k, v in LABEL2ID_PAP_NER.items()}

# ---------------- 2. HELPER FUNCTIONS ----------------
def set_random_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def _get_amp_context(use_amp: bool, device: torch.device):
    if not use_amp or not str(device).startswith('cuda'):
        return nullcontext()
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
        return torch.amp.autocast('cuda')
    return torch.cuda.amp.autocast()

def _get_grad_scaler(use_amp: bool, device: torch.device):
    if not use_amp or not str(device).startswith('cuda'):
        return None
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda')
    return torch.cuda.amp.GradScaler()

# ---------------- 3. DATA PROCESSOR & DATASET ----------------
class NerFeatures(object):
    def __init__(self, input_ids, token_type_ids, attention_mask, valid_ids, labels, label_masks):
        self.input_ids = torch.as_tensor(input_ids, dtype=torch.long)
        self.token_type_ids = torch.as_tensor(token_type_ids, dtype=torch.long)
        self.attention_mask = torch.as_tensor(attention_mask, dtype=torch.long)
        self.valid_ids = torch.as_tensor(valid_ids, dtype=torch.long)
        self.labels = torch.as_tensor(labels, dtype=torch.long)
        self.label_masks = torch.as_tensor(label_masks, dtype=torch.long)

class PapNerDataset(Dataset):
    def __init__(self, features: List[NerFeatures]):
        self.features = features
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        feat = self.features[idx]
        return {
            'input_ids': feat.input_ids,
            'token_type_ids': feat.token_type_ids,
            'attention_mask': feat.attention_mask,
            'valid_ids': feat.valid_ids,
            'labels': feat.labels,
            'label_masks': feat.label_masks
        }

def load_conll_features(data_path: str, tokenizer, max_seq_len: int = 256) -> List[NerFeatures]:
    features, tokens, tag_ids = [], [], []
    data = pd.read_csv(data_path, sep=r'\s+', encoding='utf-8', skip_blank_lines=False, names=['token', 'ner']).ffill()
    
    for row_idx, row in enumerate(data.itertuples(index=False)):
        token_val = getattr(row, 'token', None)
        ner_val = getattr(row, 'ner', None)
        if pd.notna(token_val) and str(token_val).strip() != '':
            tokens.append(str(token_val).strip().replace(' ', '_'))
            tag_ids.append(LABEL2ID_PAP_NER.get(str(ner_val).strip() if pd.notna(ner_val) else 'O', 0))
            if row_idx != len(data) - 1:
                continue
        seq_len = len(tokens)
        if seq_len == 0:
            continue
        sentence = ' '.join(tokens)
        encoding = tokenizer(sentence, padding='max_length', truncation=True, max_length=max_seq_len)
        subwords = tokenizer.tokenize(sentence)
        valid_ids = np.zeros(len(encoding['input_ids']), dtype=int)
        label_masks = np.zeros(len(encoding['input_ids']), dtype=int)
        i = 1
        for idx, subword in enumerate(subwords[:max_seq_len - 2]):
            if idx != 0 and subwords[idx - 1].endswith('@@'):
                continue
            valid_ids[i - 1] = idx + 1
            i += 1
        if max_seq_len >= seq_len:
            label_masks[:seq_len] = 1
            tag_ids.extend([0] * (max_seq_len - seq_len))
        else:
            tag_ids = tag_ids[:max_seq_len]
            label_masks[:-2] = 1
            tag_ids[-2:] = 0
        token_types = encoding.get('token_type_ids', [0] * max_seq_len)
        features.append(NerFeatures(
            input_ids=encoding['input_ids'], token_type_ids=token_types,
            attention_mask=encoding['attention_mask'], valid_ids=valid_ids,
            labels=tag_ids, label_masks=label_masks
        ))
        tokens, tag_ids = [], []
    return features

# ---------------- 4. PROJECTION HEAD & H-SUPCON LOSS ----------------
class ProjectionHead(nn.Module):
    def __init__(self, input_dim: int = 768, hidden_dim: int = 256, output_dim: int = 128):
        super(ProjectionHead, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.mlp(x), p=2, dim=-1)

class HierarchicalTokenSupConLoss(nn.Module):
    def __init__(self, temperature: float = 0.07, alpha: float = 0.3):
        super(HierarchicalTokenSupConLoss, self).__init__()
        self.temperature = temperature
        self.alpha = alpha

    def forward(self, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        if features.size(0) < 2:
            return features.new_tensor(0.0)
        features = F.normalize(features.float(), p=2, dim=1)
        similarity = torch.matmul(features, features.T) / self.temperature
        N = features.size(0)
        self_mask = torch.eye(N, device=features.device, dtype=torch.bool)
        labels_col = labels.contiguous().view(-1, 1)
        same_bio = torch.eq(labels_col, labels_col.T)
        
        entity_types = (labels_col + 1) // 2
        same_type = torch.eq(entity_types, entity_types.T)
        strong_pos = same_bio.float()
        weak_pos = (same_type & (~same_bio)).float() * self.alpha
        weight_mask = (strong_pos + weak_pos) * (~self_mask).float()
        
        similarity = similarity.masked_fill(self_mask, -1e9)
        logits_max, _ = torch.max(similarity, dim=1, keepdim=True)
        logits = similarity - logits_max.detach()
        exp_logits = torch.exp(logits) * (~self_mask).float()
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp_min(1e-12))
        log_prob = log_prob.masked_fill(self_mask, 0.0)
        
        weight_count = weight_mask.sum(dim=1)
        valid_anchor = weight_count > 0
        if not valid_anchor.any():
            return features.new_tensor(0.0)
        mean_log_prob_pos = (weight_mask * log_prob).sum(dim=1) / weight_count.clamp_min(1e-12)
        loss = -mean_log_prob_pos[valid_anchor].mean()
        return loss if torch.isfinite(loss) else features.new_tensor(0.0)

# ---------------- 5. STAGE 1 & STAGE 2 MODELS (NO CRF) ----------------
class NerOutput(object):
    def __init__(self, loss=None, logits=None, tags=None):
        self.loss = loss
        self.logits = logits
        self.tags = tags

class PhoBertSupConPretrainer(nn.Module):
    def __init__(self, model_name='vinai/phobert-base', proj_dim=128, temp=0.07, alpha=0.3):
        super(PhoBertSupConPretrainer, self).__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.roberta = AutoModel.from_pretrained(model_name, config=self.config)
        self.projection_head = ProjectionHead(input_dim=self.config.hidden_size, hidden_dim=256, output_dim=proj_dim)
        self.supcon_loss = HierarchicalTokenSupConLoss(temperature=temp, alpha=alpha)

    def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None, valid_ids=None, label_masks=None, **kwargs):
        seq_output = self.roberta(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)[0]
        batch_size, max_len, feat_dim = seq_output.shape
        range_vec = torch.arange(0, batch_size, dtype=torch.long, device=seq_output.device).unsqueeze(1)
        words_rep = seq_output[range_vec, valid_ids]
        
        valid_mask = (label_masks != 0) & (labels != 0)
        if not valid_mask.any():
            z_loss = seq_output.sum() * 0.0
            return NerOutput(loss=z_loss)
        
        proj = self.projection_head(words_rep[valid_mask])
        loss = self.supcon_loss(proj, labels[valid_mask])
        return NerOutput(loss=loss)

class PhoBertLinearProbe(nn.Module):
    def __init__(self, model_name='vinai/phobert-base', num_labels=11):
        super(PhoBertLinearProbe, self).__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.num_labels = num_labels
        self.roberta = AutoModel.from_pretrained(model_name, config=self.config)
        
        # ĐÓNG BĂNG 100% THAM SỐ BACKBONE (KHÔNG DÙNG CRF)
        for param in self.roberta.parameters():
            param.requires_grad = False
            
        self.dropout = nn.Dropout(self.config.hidden_dropout_prob)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None, valid_ids=None, label_masks=None, **kwargs):
        with torch.no_grad():
            seq_output = self.roberta(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)[0]
        batch_size, max_len, feat_dim = seq_output.shape
        range_vec = torch.arange(0, batch_size, dtype=torch.long, device=seq_output.device).unsqueeze(1)
        words_rep = seq_output[range_vec, valid_ids]
        
        logits = self.classifier(self.dropout(words_rep))  # [B, max_words, 11]
        preds = torch.argmax(logits, dim=-1)               # Suy luận trực tiếp không cần CRF
        
        tags = []
        for b in range(batch_size):
            tags.append(preds[b, :label_masks[b].sum().item()].cpu().tolist())
        loss = None
        if labels is not None:
            active = label_masks.view(-1) == 1
            loss = self.loss_fn(logits.view(-1, self.num_labels)[active], labels.view(-1)[active])
        return NerOutput(loss=loss, logits=logits, tags=tags)

# ---------------- 6. CoNLL-2003 STRICT ENTITY EVALUATOR ----------------
def split_tag(tag: str) -> Tuple[str, str]:
    if tag == 'O': return ('O', '')
    return tag.split('-', 1) if '-' in tag else ('', tag)

def is_chunk_end(prev: str, curr: str) -> bool:
    p1, t1 = split_tag(prev); p2, t2 = split_tag(curr)
    return p1 in ('E', 'S') or (p1 in ('B', 'I') and (p2 in ('B', 'S', 'O') or t1 != t2))

def is_chunk_start(prev: str, curr: str) -> bool:
    p1, t1 = split_tag(prev); p2, t2 = split_tag(curr)
    return p2 in ('B', 'S') or (p2 in ('I', 'E') and (p1 in ('O', 'E', 'S') or t1 != t2))

def evaluate_conll(true_seqs: List[str], pred_seqs: List[str]) -> Tuple[float, float, float]:
    correct_chunk, pred_chunks, gold_chunks = 0, 0, 0
    in_correct, last_true, last_pred = False, '', ''
    
    for true_tag, pred_tag in zip(true_seqs, pred_seqs):
        c_true_end = is_chunk_end(last_true, true_tag)
        c_pred_end = is_chunk_end(last_pred, pred_tag)
        c_true_start = is_chunk_start(last_true, true_tag)
        c_pred_start = is_chunk_start(last_pred, pred_tag)
        
        if in_correct:
            if c_true_end and c_pred_end and (split_tag(last_true)[1] == split_tag(last_pred)[1]):
                in_correct = False
                correct_chunk += 1
            elif c_true_end != c_pred_end or true_tag != pred_tag:
                in_correct = False
        if c_true_start and c_pred_start and (split_tag(true_tag)[1] == split_tag(pred_tag)[1]):
            in_correct = True
        if c_true_start: gold_chunks += 1
        if c_pred_start: pred_chunks += 1
        last_true, last_pred = true_tag, pred_tag
        
    if in_correct and is_chunk_end(last_true, 'O') and is_chunk_end(last_pred, 'O'):
        correct_chunk += 1
        
    prec = 0.0 if pred_chunks == 0 else correct_chunk / pred_chunks
    rec = 0.0 if gold_chunks == 0 else correct_chunk / gold_chunks
    f1 = 0.0 if (prec + rec) == 0 else (2.0 * prec * rec) / (prec + rec)
    return prec, rec, f1

print("✅ Toàn bộ Mã nguồn Model, Loss H-SupCon, Linear Probe và CoNLL Evaluator đã sẵn sàng!")


## Bước 5: Huấn luyện và Đánh giá Multi-Seed (Seeds: 42, 123, 2024)

Quy trình 2 giai đoạn được thực thi trực tiếp trên GPU RTX 3090:
- **Stage 1:** Pre-training PhoBERT với H-SupCon ($\alpha=0.3$).
- **Stage 2:** Đóng băng Backbone, chỉ huấn luyện `nn.Linear(768, 11)` (KHÔNG CÓ CRF).
- **Evaluation:** Đánh giá trên tập Test (5.162 câu).

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base', use_fast=False)

# Nạp Features CoNLL
print("Nạp dataset...")
train_feats = load_conll_features('./Dataset/train.txt', tokenizer, max_seq_len=256)
dev_feats = load_conll_features('./Dataset/dev.txt', tokenizer, max_seq_len=256)
test_feats = load_conll_features('./Dataset/test.txt', tokenizer, max_seq_len=256)
print(f"  ✅ Mẫu - Train: {len(train_feats):,} | Dev: {len(dev_feats):,} | Test: {len(test_feats):,}")

SEEDS = [42, 123, 2024]
results = []

for seed in SEEDS:
    print(f"\n{'='*70}")
    print(f"  🚀 TIẾN HÀNH THỰC NGHIỆM VỚI RANDOM SEED = {seed}")
    print(f"{'='*70}")
    set_random_seed(seed)
    output_dir = Path(f"outputs/phobert_hsupcon_seed_{seed}")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    train_loader = DataLoader(PapNerDataset(train_feats), batch_size=64, shuffle=True, pin_memory=True)
    dev_loader = DataLoader(PapNerDataset(dev_feats), batch_size=64, shuffle=False, pin_memory=True)
    test_loader = DataLoader(PapNerDataset(test_feats), batch_size=64, shuffle=False, pin_memory=True)
    
    # --- GIAI ĐOẠN 1: H-SUPCON PRE-TRAINING ---
    print("\n--- [STAGE 1] Pre-training H-SupCon (α=0.3, τ=0.07) ---")
    model_s1 = PhoBertSupConPretrainer('vinai/phobert-base', proj_dim=128, temp=0.07, alpha=0.3).to(device)
    opt_s1 = torch.optim.AdamW([
        {'params': model_s1.roberta.parameters(), 'lr': 2e-5},
        {'params': model_s1.projection_head.parameters(), 'lr': 1e-3}
    ], weight_decay=0.01)
    scaler_s1 = _get_grad_scaler(True, device)
    best_dev_loss = float('inf')
    best_enc_path = output_dir / 'best_encoder.pt'
    
    for epoch in range(1, 16):  # 15 epochs
        model_s1.train()
        tr_loss, tr_steps = 0.0, 0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            opt_s1.zero_grad()
            with _get_amp_context(True, device):
                out = model_s1(**batch)
            if out.loss.item() > 0:
                scaler_s1.scale(out.loss).backward()
                scaler_s1.step(opt_s1)
                scaler_s1.update()
                tr_loss += out.loss.item()
                tr_steps += 1
        avg_tr_loss = tr_loss / max(tr_steps, 1)
        
        # Validate Dev H-SupCon Loss
        model_s1.eval()
        d_loss, d_steps = 0.0, 0
        with torch.no_grad():
            for batch in dev_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                with _get_amp_context(True, device):
                    d_out = model_s1(**batch)
                if d_out.loss.item() > 0:
                    d_loss += d_out.loss.item()
                    d_steps += 1
        avg_d_loss = d_loss / max(d_steps, 1)
        print(f"  [Stage 1 - Epoch {epoch:02d}/15] Train Loss: {avg_tr_loss:.4f} | Dev Loss: {avg_d_loss:.4f}")
        if avg_d_loss < best_dev_loss:
            best_dev_loss = avg_d_loss
            torch.save(model_s1.roberta.state_dict(), best_enc_path)
            print(f"    ⭐ Đã lưu best encoder (Dev Loss: {best_dev_loss:.4f})")
    
    # --- GIAI ĐOẠN 2: LINEAR PROBE (FROZEN BACKBONE, NO CRF) ---
    print("\n--- [STAGE 2] Linear Probe Evaluation (Backbone Frozen, NO CRF) ---")
    model_s2 = PhoBertLinearProbe('vinai/phobert-base', num_labels=11).to(device)
    model_s2.roberta.load_state_dict(torch.load(best_enc_path, map_location=device))
    
    trainable = [p for p in model_s2.parameters() if p.requires_grad]
    opt_s2 = torch.optim.AdamW(trainable, lr=1e-3, weight_decay=0.01)
    scaler_s2 = _get_grad_scaler(True, device)
    best_dev_f1 = 0.0
    best_lp_path = output_dir / 'best_linear_probe.pt'
    
    for epoch in range(1, 21):  # 20 epochs
        model_s2.train()
        tr_loss = 0.0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            opt_s2.zero_grad()
            with _get_amp_context(True, device):
                out = model_s2(**batch)
            scaler_s2.scale(out.loss).backward()
            scaler_s2.step(opt_s2)
            scaler_s2.update()
            tr_loss += out.loss.item()
            
        # Validate Dev Strict Entity F1
        model_s2.eval()
        golds, preds = [], []
        with torch.no_grad():
            for batch in dev_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                with _get_amp_context(True, device):
                    out = model_s2(**batch)
                active = batch['label_masks'] == 1
                for b in range(len(out.tags)):
                    length = active[b].sum().item()
                    golds.extend([ID2LABEL_PAP_NER[t] for t in batch['labels'][b, :length].cpu().tolist()])
                    preds.extend([ID2LABEL_PAP_NER[t] for t in out.tags[b]])
        dev_p, dev_r, dev_f1 = evaluate_conll(golds, preds)
        print(f"  [Stage 2 - Epoch {epoch:02d}/20] Train Loss: {tr_loss/len(train_loader):.4f} | Dev Strict P: {dev_p:.4f} | R: {dev_r:.4f} | F1: {dev_f1:.4f}")
        if dev_f1 > best_dev_f1:
            best_dev_f1 = dev_f1
            torch.save(model_s2.state_dict(), best_lp_path)
            print(f"    ⭐ Đã lưu best Linear Probe (Dev Strict F1: {best_dev_f1:.4f})")
            
    # --- ĐÁNH GIÁ TRÊN TẬP TEST DEDUP (5.162 CÂU) ---
    model_s2.load_state_dict(torch.load(best_lp_path, map_location=device))
    model_s2.eval()
    test_golds, test_preds = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with _get_amp_context(True, device):
                out = model_s2(**batch)
            active = batch['label_masks'] == 1
            for b in range(len(out.tags)):
                length = active[b].sum().item()
                test_golds.extend([ID2LABEL_PAP_NER[t] for t in batch['labels'][b, :length].cpu().tolist()])
                test_preds.extend([ID2LABEL_PAP_NER[t] for t in out.tags[b]])
                
    test_p, test_r, test_f1 = evaluate_conll(test_golds, test_preds)
    print(f"\n🏁 KẾT QUẢ TEST (SEED {seed}): Strict P: {test_p*100:.2f}% | R: {test_r*100:.2f}% | F1: {test_f1*100:.2f}%")
    results.append({
        'Seed': seed,
        'Strict Precision': test_p,
        'Strict Recall': test_r,
        'Strict Micro-F1': test_f1
    })

print("\n✅ Đã hoàn thành thực nghiệm trên tất cả các Seeds!")


## Bước 6: Bảng tổng hợp kết quả Multi-Seed (Mean ± Std cho Bài báo)

In [ ]:
from prettytable import PrettyTable

df_res = pd.DataFrame(results)
prec_vals = df_res['Strict Precision'].values * 100
rec_vals = df_res['Strict Recall'].values * 100
f1_vals = df_res['Strict Micro-F1'].values * 100

table = PrettyTable()
table.field_names = ["Thử nghiệm / Seed", "Strict Precision (%)", "Strict Recall (%)", "Strict Micro-F1 (%)"]

for r in results:
    table.add_row([f"Seed {r['Seed']}", f"{r['Strict Precision']*100:.2f}", f"{r['Strict Recall']*100:.2f}", f"{r['Strict Micro-F1']*100:.2f}"])

table.add_row(["-"*18, "-"*20, "-"*18, "-"*20])
table.add_row([
    f"TRUNG BÌNH (μ ± σ)",
    f"{np.mean(prec_vals):.2f} ± {np.std(prec_vals):.2f}",
    f"{np.mean(rec_vals):.2f} ± {np.std(rec_vals):.2f}",
    f"{np.mean(f1_vals):.2f} ± {np.std(f1_vals):.2f}"
])

print(table)

baseline_flat_f1 = 90.02
gain = np.mean(f1_vals) - baseline_flat_f1
print(f"\n📊 SO SÁNH VỚI BASELINE FLAT SUPCON (90.02%):")
print(f"  ➡️ Mức tăng trưởng nhờ Hierarchical SupCon (α=0.3): {'+' if gain > 0 else ''}{gain:.2f}% F1!")
